# Pollinator pipeline (master notebook)

Runs end-to-end on Google Colab or locally. Each cell auto-detects the environment.

**Colab defaults:**
- Base: `/content/drive/MyDrive/aea` (set `AEA_BASE` to override)
- Extract destination: `/content/data` (set `AEA_EXTRACT` to override)
- Pipeline package: `<BASE>/ml-pipelines` (set `AEA_PIPELINE_ROOT` to override)

**Local defaults:**
- Base: `~/aea`
- Extract destination: `~/aea/extracted`
- Pipeline package: `~/aea/ml-pipelines`

Locally, set env vars before launching Jupyter, e.g.:
```
export AEA_BASE=~/data/aea
export AEA_PIPELINE_ROOT=~/repo/aea-refactor-pollinator/ml-pipelines
```

**Setup:**
- Colab: Runtime > Change runtime type > T4 GPU (or better).
- Local: ensure CUDA is available; install ultralytics + sahi in your env if not already there.


## Run mode

`SMOKE_TEST = True` runs every training stage for 1 epoch so you can verify the pipeline end-to-end before committing to a full run.


In [ ]:
SMOKE_TEST = False


## Install Colab deps

Only runs on Colab. Locally, install ultralytics + sahi via your project's env.


In [ ]:
# Skip this cell when running locally if ultralytics+sahi are already installed in your env.
import sys
if 'google.colab' in sys.modules:
    !pip install -q ultralytics sahi


## 1. Train YOLO detector

Two-stage YOLO. Output: stage2/weights/best.pt.


In [ ]:
"""Train the YOLO pollinator detector on Google Colab.

Reads yolo.zip from Drive (containing data.yaml, images/{train,val,test}/,
labels/{train,val,test}/). Copies the zip to local SSD and extracts there
because Drive FUSE I/O is slow per-file across many small files.

After extracting, labels are patched:
- Lines for classes not in KEEP_CLASSES are filtered out; remaining
  indices are remapped to 0..N-1.
- Now-empty label files plus their orphan images are removed.
- data.yaml is rewritten with the KEEP_CLASSES names.

MERGE_TEST_INTO_TRAIN moves the test split into train before patching.
Use this in Colab to maximise training data. In production retraining
(via the backend's TrainingJob), the proper train/val/test split is used.

To re-train as an incremental fine-tune from prior weights (skip the
frozen-backbone stage), set EPOCHS_STAGE1 = 0 and point MODEL_SIZE at
the existing stage1_frozen/weights/best.pt.

Adjust the constants below to match your Drive layout, then run.
"""

import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml-pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

YOLO_ZIP_PATH = f'{BASE_DIR}/datasets/yolo.zip'
EXTRACTED_SUBDIR = 'yolo'
OUTPUT_DIR = f'{BASE_DIR}/runs/yolo'

# Class layout as exported by CVAT. Order must match the class indices in
# the label .txt files (one class per index, starting at 0). Include every
# class CVAT produces here, even ones you want to drop (like 'unsure').
CVAT_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'unsure']

# Subset of CVAT_CLASSES to train on. Anything not in this list is dropped
# from labels; remaining indices are remapped to 0..N-1.
KEEP_CLASSES = ['fly', 'butterfly']

# Move test/ into train/ before training (maximum training data, val stays
# as the held-out set). Set False to keep the three-way split that the
# backend's TrainingJob retraining uses.
MERGE_TEST_INTO_TRAIN = True

# Smoke mode: when True, both stages run 1 epoch for an end-to-end check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters.
MODEL_SIZE = 'yolo26n.pt'
IMG_SIZE = 1024
BATCH = 16
EPOCHS_STAGE1 = 1 if SMOKE_TEST else 30
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 70
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
SEED = 42


def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip to local SSD and unzip into <extract_to>/<expected_subdir>/."""
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def merge_test_into_train(dataset_root: str) -> None:
    """Move test/* into train/* for both images/ and labels/. Idempotent:
    no-op if the test split is already gone."""
    import shutil
    from pathlib import Path

    root = Path(dataset_root)
    moved = 0
    for subdir in ('images', 'labels'):
        test_dir = root / subdir / 'test'
        train_dir = root / subdir / 'train'
        if not test_dir.exists():
            continue
        train_dir.mkdir(parents=True, exist_ok=True)
        for item in list(test_dir.iterdir()):
            shutil.move(str(item), str(train_dir / item.name))
            moved += 1
        test_dir.rmdir()
    if moved:
        print(f'[merge] moved {moved} test files into train/')
    else:
        print('[merge] no test files to move (already merged or no test split)')


def _write_data_yaml(dataset_root, names) -> None:
    from pathlib import Path
    root = Path(dataset_root)
    lines = [f'path: {root}']
    for split in ('train', 'val', 'test'):
        if (root / 'images' / split).exists():
            lines.append(f'{split}: images/{split}')
    lines.append('names:')
    for i, name in enumerate(names):
        lines.append(f'  {i}: {name}')
    (root / 'data.yaml').write_text('\n'.join(lines) + '\n')
    print(f'[patch] rewrote {root / "data.yaml"} with {len(names)} classes')


def patch_dataset(dataset_root: str, cvat_classes: list, keep_classes: list) -> None:
    """Filter labels to keep_classes, remap indices to 0..N-1, drop now-empty
    labels and orphan images, then rewrite data.yaml.

    Idempotent via a .patched_for marker.
    """
    from pathlib import Path

    root = Path(dataset_root)
    marker = root / '.patched_for'
    config_id = ','.join(keep_classes)

    if marker.exists() and marker.read_text() == config_id:
        print(f'[patch] dataset already patched for {keep_classes}; skipping')
        _write_data_yaml(root, keep_classes)
        return

    cvat_to_idx = {name: i for i, name in enumerate(cvat_classes)}
    remap = {
        cvat_to_idx[name]: new_idx
        for new_idx, name in enumerate(keep_classes)
        if name in cvat_to_idx
    }

    n_lines_stripped = 0
    n_labels_removed = 0
    n_images_removed = 0

    for split in ('train', 'val', 'test'):
        labels_dir = root / 'labels' / split
        images_dir = root / 'images' / split
        if not labels_dir.exists():
            continue
        for label_file in labels_dir.glob('*.txt'):
            original = label_file.read_text().splitlines()
            kept = []
            for line in original:
                parts = line.strip().split()
                if not parts:
                    continue
                try:
                    cls = int(parts[0])
                except ValueError:
                    continue
                if cls not in remap:
                    continue
                parts[0] = str(remap[cls])
                kept.append(' '.join(parts))
            n_lines_stripped += len(original) - len(kept)
            if kept:
                label_file.write_text('\n'.join(kept) + '\n')
            else:
                label_file.unlink()
                n_labels_removed += 1
                if images_dir.exists():
                    for img in images_dir.glob(f'{label_file.stem}.*'):
                        img.unlink()
                        n_images_removed += 1

    print(
        f'[patch] keep={keep_classes}: stripped {n_lines_stripped} lines, '
        f'removed {n_labels_removed} empty labels and {n_images_removed} orphan images'
    )

    _write_data_yaml(root, keep_classes)
    marker.write_text(config_id)


def main() -> None:
    import shutil
    from pathlib import Path

    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    dataset_root = ensure_extracted(
        YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR,
    )

    # Config that requires a fresh extraction if it changed: KEEP_CLASSES
    # (label-remap state) and MERGE_TEST_INTO_TRAIN (test split removed).
    marker = Path(dataset_root) / '.patched_for'
    config_id = f"keep={','.join(KEEP_CLASSES)}|merge_test={MERGE_TEST_INTO_TRAIN}"
    if marker.exists() and marker.read_text() != config_id:
        print('[setup] config changed since last run; re-extracting from zip...')
        shutil.rmtree(dataset_root)
        dataset_root = ensure_extracted(
            YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR,
        )

    if MERGE_TEST_INTO_TRAIN:
        merge_test_into_train(dataset_root)

    patch_dataset(dataset_root, cvat_classes=CVAT_CLASSES, keep_classes=KEEP_CLASSES)

    # Mark the dataset with the full config so the next run can detect changes.
    (Path(dataset_root) / '.patched_for').write_text(config_id)

    from pollinator.training import train_yolo

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] {processed}/{total} {message}')

    result = train_yolo(
        dataset_root=dataset_root,
        output_dir=OUTPUT_DIR,
        model_size=MODEL_SIZE,
        img_size=IMG_SIZE,
        batch=BATCH,
        epochs_stage1=EPOCHS_STAGE1,
        epochs_stage2=EPOCHS_STAGE2,
        lr_stage1=LR_STAGE1,
        lr_stage2=LR_STAGE2,
        classes=KEEP_CLASSES,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


## 2. Train EfficientNet binary classifier

Output: binary_best.pth.


In [ ]:
"""Train the EfficientNet binary insect/background classifier on Colab.

Reads `annotated_crops.zip` from Drive (containing labeled_ls/ and
labeled_mb/, each with bumblebee, fly, butterfly, other, background
subdirs). Copies the zip to local SSD and extracts there because Drive
FUSE I/O is slow per-file; one big copy plus a local unzip is dramatically
faster than reading the unzipped tree off Drive.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml-pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml-pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

ANNOTATED_ZIP_PATH = f'{BASE_DIR}/datasets/annotated_crops.zip'
OUTPUT_DIR = f'{BASE_DIR}/runs/binary'

# Model. 'efficientnet' uses torchvision-pretrained EfficientNet-B2.
# 'insectnet' loads INSECTNET_WEIGHTS as a backbone init.
MODEL_TYPE = 'efficientnet'
INSECTNET_WEIGHTS = f'{BASE_DIR}/weights/insectnet_pretrained.pth'

# Smoke mode: see train_yolo.py for the full explanation. When True,
# epochs collapse to 1 for a fast end-to-end sanity check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters.
EPOCHS = 1 if SMOKE_TEST else 20
BATCH = 32
LR = 1e-3
VAL_FRAC = 0.2
TEST_FRAC = 0.1
BG_RATIO = 3
SEED = 42

def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip from Drive to local SSD and unzip into <extract_to>/<expected_subdir>/.

    Handles zips with or without a top-level wrapping directory. If the zip
    wraps its content in a same-named dir, the contents are flattened up one
    level so the returned path always contains the actual data.
    """
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    annotated_root = ensure_extracted(
        ANNOTATED_ZIP_PATH, LOCAL_EXTRACT_DIR, 'annotated_crops',
    )
    data_dirs = [
        f'{annotated_root}/labeled_ls',
        f'{annotated_root}/labeled_mb',
    ]

    from pollinator.training import train_binary

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] epoch {processed}/{total} {message}')

    result = train_binary(
        data_dirs=data_dirs,
        model_type=MODEL_TYPE,
        insectnet_weights=INSECTNET_WEIGHTS if MODEL_TYPE == 'insectnet' else None,
        output_dir=OUTPUT_DIR,
        epochs=EPOCHS,
        batch=BATCH,
        lr=LR,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
        bg_ratio=BG_RATIO,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


## 3. Train InsectNet group classifier

Output: group_best.pth.


In [ ]:
"""Train the group classifier (bumblebee/fly/butterfly/other) on Colab.

Reuses `annotated_crops.zip` (the four insect subdirs; background is
ignored) and adds `web_images.zip` for iNaturalist-style augmentation.

Two stages: stage 1 trains the head, stage 2 (optional) unfreezes the
last block. EPOCHS_S2=0 is recommended for InsectNet on small Arctic
data since stage 2 tends to overfit.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml-pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml-pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

ANNOTATED_ZIP_PATH = f'{BASE_DIR}/datasets/annotated_crops.zip'
WEB_ZIP_PATH = f'{BASE_DIR}/datasets/web_images.zip'
OUTPUT_DIR = f'{BASE_DIR}/runs/group'

# Model. 'insectnet' needs INSECTNET_WEIGHTS; 'efficientnet' does not.
MODEL_TYPE = 'insectnet'
INSECTNET_WEIGHTS = f'{BASE_DIR}/weights/insectnet_pretrained.pth'

# Smoke mode: see train_yolo.py for the full explanation. When True,
# epochs collapse to 1 for a fast end-to-end sanity check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters.
EPOCHS_S1 = 1 if SMOKE_TEST else 20
EPOCHS_S2 = 0
BATCH = 32
LR_S1 = 1e-3
LR_S2 = 1e-4
VAL_FRAC = 0.2
TEST_FRAC = 0.1
SEED = 42

def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip from Drive to local SSD and unzip into <extract_to>/<expected_subdir>/.

    Handles zips with or without a top-level wrapping directory. If the zip
    wraps its content in a same-named dir, the contents are flattened up one
    level so the returned path always contains the actual data.
    """
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    annotated_root = ensure_extracted(
        ANNOTATED_ZIP_PATH, LOCAL_EXTRACT_DIR, 'annotated_crops',
    )
    web_root = ensure_extracted(
        WEB_ZIP_PATH, LOCAL_EXTRACT_DIR, 'web_images',
    )
    data_dirs = [
        f'{annotated_root}/labeled_ls',
        f'{annotated_root}/labeled_mb',
    ]
    web_dir = web_root

    from pollinator.training import train_group

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] epoch {processed}/{total} {message}')

    result = train_group(
        data_dirs=data_dirs,
        web_dir=web_dir,
        model_type=MODEL_TYPE,
        insectnet_weights=INSECTNET_WEIGHTS if MODEL_TYPE == 'insectnet' else None,
        output_dir=OUTPUT_DIR,
        epochs_s1=EPOCHS_S1,
        epochs_s2=EPOCHS_S2,
        batch=BATCH,
        lr_s1=LR_S1,
        lr_s2=LR_S2,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


## 4. Run inference on a camera folder

Output: results.json.


In [ ]:
"""Run the full pollinator inference pipeline on Colab.

Processes one camera-plot folder end-to-end: YOLO detection in parallel
with classical preprocessing, then InsectNet binary gate, then group
classifier, then merge by IoU. Writes results.json plus per-detection
crops under OUTPUT_DIR.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml-pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml-pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

IMAGE_DIR = f'{BASE_DIR}/inference/camera_a'
OUTPUT_DIR = f'{BASE_DIR}/runs/inference/camera_a'

# Model checkpoints.
YOLO_MODEL = f'{BASE_DIR}/runs/yolo/stage2/weights/best.pt'
BINARY_MODEL = f'{BASE_DIR}/runs/binary/binary_best.pth'
GROUP_MODEL = f'{BASE_DIR}/runs/group/group_best.pth'

# Knobs that the upload UI also exposes. Defaults match the production
# pipeline; override per-camera if needed.
YOLO_CONFIDENCE = 0.4
BINARY_THRESHOLD = 0.5
IOU_THRESHOLD = 0.3
SKIP_FIRST_N = 0
DEBUG = False

PREPROCESSING_CONFIG = {
    'crop_pad_frac': 0.3,
    'background_sample_size': 100,
    'min_contour_area': 400,
    'max_contour_area': 35000,
    'sunny_shutter_threshold': 150,
    'skip_flash': True,
    'skip_foggy': True,
    'enable_large_motion': True,
}

def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    from pollinator.inference import run_pipeline

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] {processed}/{total} {message}')

    result = run_pipeline(
        image_dir=IMAGE_DIR,
        output_dir=OUTPUT_DIR,
        yolo_model=YOLO_MODEL,
        binary_model=BINARY_MODEL,
        group_model=GROUP_MODEL,
        config=PREPROCESSING_CONFIG,
        yolo_confidence=YOLO_CONFIDENCE,
        binary_threshold=BINARY_THRESHOLD,
        iou_threshold=IOU_THRESHOLD,
        skip_first_n=SKIP_FIRST_N,
        debug=DEBUG,
        progress_callback=on_progress,
    )

    print('\n=== Inference complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()
